# OpenScan OCR | Laboratorio público

**De imágenes escaneadas a texto, con resultados medibles.**

Este cuaderno acompaña al proyecto abierto [OpenScan OCR](https://github.com/isyoimerr/openscan-ocr). Explica un flujo real, reproducible y sin necesidad de subir documentos propios.

**Qué incluye:** Tesseract local, preprocesamiento, CER/WER, PSM, PDF con texto buscable y una comparación opcional con Gemini. Todos los ejemplos son **ficticios**.

> Ejecuta las celdas en orden. Las funciones de la app también están disponibles en `src/openscan/`.

In [ ]:
# 1. Preparar el entorno (Colab o equipo propio)
import os, sys, shutil, subprocess
from pathlib import Path

if 'google.colab' in sys.modules:
    subprocess.run(['apt-get', '-qq', 'install', '-y',
                    'tesseract-ocr', 'tesseract-ocr-spa'], check=True)
elif shutil.which('tesseract') is None:
    raise RuntimeError('Instala primero Tesseract OCR. Consulta docs/INSTALL.md')

subprocess.run([
    sys.executable, '-m', 'pip', '-q', 'install',
    'Pillow>=10', 'numpy<3', 'opencv-python-headless', 'pytesseract',
    'pypdfium2', 'pypdf', 'pandas', 'matplotlib', 'google-genai'
], check=True)
print('Dependencias listas')

In [ ]:
# 2. Obtener el codigo publico y seleccionar las muestras
import sys, subprocess
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src" / "openscan").exists():
    if (ROOT.parent / "src" / "openscan").exists():
        ROOT = ROOT.parent
    else:
        ROOT = Path("/content/openscan-ocr")
        if not (ROOT / "src" / "openscan").exists():
            subprocess.run([
                "git", "clone", "--depth", "1",
                "https://github.com/isyoimerr/openscan-ocr.git", str(ROOT)
            ], check=True)
sys.path.insert(0, str(ROOT / "src"))

from openscan.documents import load_pages
from openscan.preprocessing import preprocess_image
from openscan.pipeline import process_page
from openscan.engines import run_tesseract, run_gemini, GEMINI_DEFAULT_MODEL
from openscan.export import images_to_searchable_pdf
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

EXAMPLES = ROOT / "examples"
reference = (EXAMPLES / "factura_referencia.txt").read_text(encoding="utf-8")
image_paths = [
    EXAMPLES / "factura_01_limpia.png",
    EXAMPLES / "factura_02_inclinada.png",
    EXAMPLES / "factura_03_ruido.png",
]
pages = [load_pages(p.name, p.read_bytes())[0] for p in image_paths]
print("Muestras cargadas:", len(pages))

## 1. ¿Qué lee el motor?

Cada imagen contiene **la misma factura ficticia**, pero con calidad diferente. Para evaluar la precisión disponemos del texto original en `factura_referencia.txt`.

Si quieres probar una imagen tuya, cambia las rutas en `image_paths`. Para medir CER o WER necesitarás su transcripción correcta: no se deben asignar porcentajes inventados.

In [ ]:
# 3. Vista previa y OCR local sobre una imagen
imagen = pages[1].image
vista = preprocess_image(imagen, "otsu")
fig, axes = plt.subplots(1, 2, figsize=(11, 6))
axes[0].imshow(imagen); axes[0].set_title("Imagen original"); axes[0].axis("off")
axes[1].imshow(vista, cmap="gray"); axes[1].set_title("Otsu"); axes[1].axis("off")
plt.tight_layout(); plt.show()

row, result = process_page(pages[1], preprocessing="original", psm=6,
                           lang="spa", reference=reference)
print("Motor:", result.engine, "| CER:", round(row["cer"], 4),
      "| Tiempo:", round(row["tiempo_segundos"], 2), "s")
print("\nTexto extraido:\n", result.text[:800])

## 2. Benchmark reproducible: binarización y PSM

Probamos **27 combinaciones**: 3 imágenes × 3 tratamientos (original, Otsu, adaptativa) × 3 modos de segmentación (4, 6, 11). Un CER más bajo significa menos diferencias con el texto de referencia. El rendimiento depende de la clase de documento.

In [ ]:
# 4. Comparacion completa (sin utilizar una API externa)
rows = []
for page in pages:
    for mode in ["original", "otsu", "adaptive"]:
        for psm in [4, 6, 11]:
            row, _ = process_page(page, preprocessing=mode,
                                  psm=psm, lang="spa", reference=reference)
            rows.append(row)
benchmark = pd.DataFrame(rows)
print("Combinaciones evaluadas:", len(benchmark))
display(benchmark[["archivo", "preprocesamiento", "psm", "cer", "wer", "tiempo_segundos"]].round(4))
print("CER medio por preprocesamiento:")
display(benchmark.groupby("preprocesamiento")["cer"].mean().round(4))
print("CER medio por PSM:")
display(benchmark.groupby("psm")["cer"].mean().round(4))
benchmark.groupby("psm")["cer"].mean().plot(kind="bar", ylabel="CER medio", title="Tasa de error por PSM")
plt.show()

## 3. PDF escaneado → PDF buscable

Un PDF construido a partir de fotos normalmente no permite seleccionar palabras. Aplicamos Tesseract y generamos una **capa de texto invisible** sobre cada página. El archivo procesado queda en la carpeta `openscan_resultados`.

In [ ]:
# 5. Extraer paginas y generar PDF seleccionable
pdf_path = EXAMPLES / "facturas_demo_3paginas.pdf"
pdf_pages = load_pages(pdf_path.name, pdf_path.read_bytes(), max_pages=5)
OUTPUT = ROOT / "openscan_resultados"
OUTPUT.mkdir(exist_ok=True)

textos = []
for page in pdf_pages:
    item = run_tesseract(page.image, lang="spa", psm=6)
    textos.append(item.text)
    print("Pagina", page.page_number, "| caracteres:", len(item.text))

(OUTPUT / "texto_extraido.txt").write_text("\n\n--- PAGINA ---\n\n".join(textos), encoding="utf-8")
(OUTPUT / "documento_buscable.pdf").write_bytes(
    images_to_searchable_pdf([page.image for page in pdf_pages], lang="spa", psm=6)
)
print("Archivos guardados en:", OUTPUT)

## 4. Gemini (opcional): comparar OCR local y nube

Gemini envía la imagen a una API externa; **no es necesario para ejecutar el resto del cuaderno**. Si quieres probarlo en Colab, guarda tu clave en la sección **Secretos** con el nombre `API`, sin mostrarla ni pegarla dentro de una celda. Puede haber cuotas o errores `503`.

In [ ]:
# 6. Comparar con Gemini solo si decides activarlo
USAR_GEMINI = False
MODELO_GEMINI = GEMINI_DEFAULT_MODEL
comparacion = benchmark.copy()

if USAR_GEMINI:
    try:
        from google.colab import userdata
        api_key = userdata.get("API")
    except ImportError:
        api_key = os.environ.get("GEMINI_API_KEY")
    if not api_key:
        print("Gemini omitido: no se encontro el secreto API.")
    else:
        try:
            row, output = process_page(
                pages[1], engine="Gemini", preprocessing="original",
                reference=reference, gemini_key=api_key, gemini_model=MODELO_GEMINI
            )
            print("Gemini:", output.model, "CER:", round(row["cer"], 4),
                  "Tiempo:", round(row["tiempo_segundos"], 2), "s")
            print("Tokens de entrada:", output.input_tokens,
                  "| salida:", output.output_tokens)
            comparacion = pd.concat([comparacion, pd.DataFrame([row])], ignore_index=True)
        except Exception as exc:
            print("Gemini no disponible:", str(exc)[:160])
else:
    print("Gemini desactivado. El laboratorio ya funciona con Tesseract.")

In [ ]:
# 7. Guardar CSV y descargar archivos en Colab
csv_path = OUTPUT / "resultados_ocr.csv"
comparacion.to_csv(csv_path, index=False, encoding="utf-8-sig")
print("CSV:", csv_path)
print("PDF buscable:", OUTPUT / "documento_buscable.pdf")

# Para descargar automaticamente, cambia la siguiente variable a True.
DESCARGAR = False
if DESCARGAR:
    try:
        from google.colab import files
        files.download(str(csv_path))
        files.download(str(OUTPUT / "documento_buscable.pdf"))
    except ImportError:
        print("Descargas automaticas disponibles solo en Colab.")

## Interpretación y siguientes pasos

- No existe un tratamiento que siempre mejore todas las imágenes: comprueba el CER según el documento.
- Los modos PSM cambian cómo Tesseract entiende la distribución del texto.
- No se puede afirmar que un OCR es 100 % preciso sin un texto de referencia y un conjunto de validación representativo.
- El prototipo para usuarios no técnicos está en `app/streamlit_app.py`.

**¿Quieres contribuir?** Revisa el [README](https://github.com/isyoimerr/openscan-ocr) y abre un issue con una prueba, una mejora del pipeline o una propuesta de interfaz.